# ChatPromptTemplate的高级特性

## 1.部分变量预填充：partial()

举例：

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# 原始模板
template = ChatPromptTemplate.from_messages([
    ("system", "你是{role},目标用户是{audience}"),
    ("user", "{task}")
])

result1 = template.invoke({"role":"导游","audience":"游客","task":"介绍一下北京的故宫"})
result2 = template.invoke({"role":"导游","audience":"游客","task":"介绍一下北京的天坛"})

print(result1)
print(result2)

In [ ]:
上述代码可以使用partial()

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# 原始模板
template = ChatPromptTemplate.from_messages([
    ("system", "你是{role},目标用户是{audience}"),
    ("user", "{task}")
])

# 部分变量预填充
final_template = template.partial(role="导游",audience = "游客")

result1 = final_template.invoke({"task":"介绍一下北京的故宫"})
result2 = final_template.invoke({"task":"介绍一下北京的天坛"})

print(result1)
print(result2)

举例：

In [ ]:
# 场景：为不同部门创建专用模板
base_template = ChatPromptTemplate.from_messages([
    ("system", "你是{department}的{role}"),
    ("user", "{task}")
])

# IT 部门
it_template = base_template.partial(
    department="IT 部门",
    role="技术支持"
)

# 销售部门
sales_template = base_template.partial(
    department="销售部门",
    role="销售顾问"
)

sales_template.invoke({"task":"为什么每年年底汽车会促销"})

## 2.使用消息占位符

### 2.1使用 placeholder

举例：

In [ ]:
template = ChatPromptTemplate.from_messages([
    ("system","我是一个AI助手"),
    ("placeholder","{conversation}")
])

res = template.invoke({
    "conversation": [
        ("human","你好，请问明天天气如何?"),
        ("ai","明天天气晴朗"),
        ("human","后天天气如何")
    ]
})

print(res)

### 2.2.使用MessagesPlaceholder

举例：

In [ ]:
template = ChatPromptTemplate.from_messages([
    ("system","我是一个AI助手"),
    MessagesPlaceholder(variable_name = "conversation")
])

res = template.invoke({
    "conversation": [
        ("human","你好，请问明天天气如何?"),
        ("ai","明天天气晴朗"),
        ("human","后天天气如何")
    ]
})

print(res)

举例：存储历史消息

In [9]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt_template = ChatPromptTemplate([
    ("system", "你是一个友好的AI助手"),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{question}")
])

prompt_template.invoke({
    "history": [
        ("human", "5 + 2 = ?"),
        ("ai", "5 + 2 = 7")
    ],
    "question":"结果再乘以4呢？"
})

ChatPromptValue(messages=[SystemMessage(content='你是一个友好的AI助手', additional_kwargs={}, response_metadata={}), HumanMessage(content='5 + 2 = ?', additional_kwargs={}, response_metadata={}), AIMessage(content='5 + 2 = 7', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='结果再乘以4呢？', additional_kwargs={}, response_metadata={})])

### 2.3可复用的模板库

定义了具体的存放模板的文件

In [10]:
from langchain_core.prompts import ChatPromptTemplate

class PromptLibrary:
    """可复用的提示词模板库"""

    TRANSLATOR = ChatPromptTemplate.from_messages([
        ("system", "你是专业翻译，精通{source_lang}和{target_lang}"),
        ("user", "翻译以下文本：\n{text}")
    ])

    CODE_REVIEWER = ChatPromptTemplate.from_messages([
        ("system", "你是{language}代码审查专家，重点关注{focus}"),
        ("user", "审查代码：\n```{language}\n{code}\n```")
    ])

    SUMMARIZER = ChatPromptTemplate.from_messages([
        ("system", "你是内容摘要专家"),
        ("user", "将以下内容总结为{num}个要点：\n{content}")
    ])

    TUTOR = ChatPromptTemplate.from_messages([
        ("system", "你是{subject}导师，学生水平：{level}"),
        ("user", "{question}")
    ])

其他文件中进行调用

In [11]:
# from templates import PromptLibrary

messages = PromptLibrary.TRANSLATOR.format_messages(
    source_lang = "英语",
    target_lang = "中文",
    text = "Hello World"
)